# Module 5: Predictor Calculation: Proof of Concept & Implementation Blueprint

## Overview
This Jupyter Notebook serves as the **executable specification and Proof of Concept** for **Module 5: Predictor Calculation** in the Luma Geospatial Engine (`luma-ge`). It provides the engineering reference for the backend (`luma-be`) and frontend (`luma-fe`).

### Live Catalog Status
- **Total Predictors Supported**: 30
- **Spectral Indices Supported**: 20 (NDVI, EVI, SAVI, MSAVI, GNDVI, MCARI, RENDVI, MCARI1, MNDWI, NDMI, AWEIsh, AWEInsh, NDTI, ANDWI, NDBI, DBSI, BLFEI, MBI, EBBI)
- **Topographical data Supported**
- **Categories Covered**: Vegetation, Water, Built-up & Bare Soil, Topography, Radar Polarization

### Key Features Demonstrated:
1. **Sensor-Aware Predictor Catalog**: Dynamic filtering based on optical sensor (`LANDSAT` vs. `SENTINEL-2`) and temporal windows.
2. **Spectral Indices Engine**: Automated computation via `spyndex` and `luma-ge.predictor`.
3. **Terrain Predictors**: Elevation, Slope, Aspect, and Landform morphometry derived from DEMs (`NASADEM`, `ALOS`).
4. **SAR Radar Predictors**: Sentinel-1 C-band ($VV$, $VH$, ratio) and ALOS PALSAR / PALSAR-2 L-band ($HH$, $HV$, ratio).
5. **End-to-End Orchestration**: High-level execution via `PredictorCalculation.compute_predictors()`, maintaining complete contract compatibility with `luma-be`.
6. **Interactive Visualization**: Multi-layer rendering using `geemap`.
7. **BE / FE Contract Specs**: Complete JSON payloads and layer styling palettes for direct web application implementation.

---
## 1. Environment Setup & Earth Engine Initialization
Initialize Google Earth Engine using `luma-ge`'s lazy initialization mechanism and import required geospatial libraries.

In [1]:
import ee
import geemap
import pandas as pd
from luma_ge.ee_config import ensure_ee_initialized
from luma_ge.data_acquisition import Reflectance_Data, final_Image
from luma_ge.predictor import (
    PredictorCalculation,
    predictor_catalog,
    SpectralCalculator,
    sensor_detector,
    terrain_calculator
)
from luma_ge.sar import get_sar_data

# Initialize Earth Engine
ensure_ee_initialized()
print("Google Earth Engine initialized successfully!")

2026-11-10 15:58:29,335 - luma_ge.ee_config - INFO - Earth Engine initialized successfully


Google Earth Engine initialized successfully!


---
## 2. Area of Interest (AOI) & Time Window
We define a representative testing region in **South Sumatra, Indonesia** featuring a diverse mix of:
- Dense natural & secondary forest
- Agricultural plantations (oil palm / rubber)
- Peatlands, inland water bodies, and coastal wetlands
- Urban / built-up settlements

In [2]:
# Define Area of Interest (AOI) - Banyuasin / Palembang, South Sumatra
aoi = ee.Geometry.Rectangle([104.60, -3.10, 104.90, -2.80])

# Temporal window: Annual composite window for 2021
start_date = '2021-01-01'
end_date = '2021-12-31'

print("AOI Coordinates:", aoi.coordinates().getInfo())
print(f"Temporal Window: {start_date} to {end_date}")

AOI Coordinates: [[[104.6, -3.1], [104.9, -3.1], [104.9, -2.8], [104.6, -2.8], [104.6, -3.1]]]
Temporal Window: 2021-01-01 to 2021-12-31


---
## 3. Predictor Catalog & Sensor-Aware Availability Check
`PredictorCatalog` dynamically defines compatibility rules for all predictors.
- **Landsat-Specific**: e.g., `EBBI` (requires Thermal and SWIR).
- **Sentinel-2 Specific**: e.g., `RENDVI`, `MCARI` (require Red-Edge bands).
- **Radar**: Sentinel-1 ($VV$, $VH$ available from Oct 2014) vs. ALOS PALSAR ($HH$, $HV$).

The frontend calls the backend, which queries `PredictorCalculation.get_available_predictors()` to know which indices to display or grey out.

In [3]:
predictor_service = PredictorCalculation()

# Query available predictors for Sentinel-2
s2_catalog = predictor_service.get_available_predictors(
    start_date=start_date,
    end_date=end_date,
    sensor='SENTINEL'
)

# Query available predictors for Landsat
landsat_catalog = predictor_service.get_available_predictors(
    start_date=start_date,
    end_date=end_date,
    sensor='LANDSAT'
)

# Access the inner predictors dictionaries
s2_preds = s2_catalog.get('predictors', {})
landsat_preds = landsat_catalog.get('predictors', {})

# Convert to a comparison DataFrame for inspection
df_comparison = pd.DataFrame([
    {
        "Predictor": name,
        "Category": meta.get('category', ''),
        "Sensor Target": meta.get('sensor_compatibility', meta.get('sensor', 'ALL')),
        "Sentinel-2": "Available" if s2_preds.get(name, {}).get('available') else f"Excluded: {s2_preds.get(name, {}).get('reason')}",
        "Landsat": "Available" if landsat_preds.get(name, {}).get('available') else f"Excluded: {landsat_preds.get(name, {}).get('reason')}"
    }
    for name, meta in s2_preds.items()
])

print(f"Total Predictors in Catalog: {len(df_comparison)}")
df_comparison

2026-11-10 15:58:30,047 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-11-10 15:58:31,580 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-11-10 15:58:31,581 - luma_ge.sar - INFO - get_sar_data initialized
2026-11-10 15:58:31,583 - luma_ge.predictor - INFO - Predictor Calculation initialized with terrain, spectral and SAR calculators


Total Predictors in Catalog: 29


,Predictor,Category,Sensor Target,Sentinel-2,Landsat
0,NDVI,Vegetasi,ALL,Available,Available
1,EVI,Vegetasi,ALL,Available,Available
2,SAVI,Vegetasi,ALL,Available,Available
3,MSAVI,Vegetasi,ALL,Available,Available
4,GNDVI,Vegetasi,ALL,Available,Available
5,MCARI,Vegetasi,SENTINEL,Available,Excluded: Requires Sentinel-2 Red-Edge band (RE1)
6,RENDVI,Vegetasi,SENTINEL,Available,Excluded: Requires Sentinel-2 Red-Edge bands (...
7,MCARI1,Vegetasi,ALL,Available,Available
8,MNDWI,Air & Kelembaban,ALL,Available,Available
9,NDMI,Air & Kelembaban,ALL,Available,Available


---
## 4. Optical Imagery Acquisition (Module 1 Handshake)
We acquire standard cloud-masked, harmonized optical composites:
1. **Sentinel-2 Level-2A** via `ref_data.get_s2_optical_data()`
2. **Landsat 8 Level-2** via `ref_data.get_optical_data()`

In [4]:
ref_data = Reflectance_Data()
processor = final_Image()

# 1. Acquire Sentinel-2 Surface Reflectance Collection & Composite
print("Fetching Sentinel-2 collection...")
s2_col, s2_stats = ref_data.get_s2_optical_data(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    cloud_cover=40,
    compute_detailed_stats=False
)
s2_comp_res = processor.get_temporal_composite(s2_col, aoi, reducer='median')
s2_composite = s2_comp_res['image']
print("Sentinel-2 Bands:", s2_composite.bandNames().getInfo())

# 2. Acquire Landsat 8 Surface Reflectance Collection & Composite
print("\nFetching Landsat 8 collection...")
l8_col, l8_stats = ref_data.get_optical_data(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    optical_data='L8_SR',
    cloud_cover=40,
    compute_detailed_stats=False
)
l8_comp_res = processor.get_temporal_composite(l8_col, aoi, reducer='median')
l8_composite = l8_comp_res['image']
print("Landsat 8 Bands:", l8_composite.bandNames().getInfo())

2026-11-10 15:58:44,205 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-11-10 15:58:44,206 - final_Image - INFO - final_Image creation initialized.
2026-11-10 15:58:44,207 - Reflectance_Data - INFO - Date range: 2021-01-01 to 2021-12-31
2026-11-10 15:58:44,207 - Reflectance_Data - INFO - Cloud cover threshold (image-level): 40%
2026-11-10 15:58:44,209 - Reflectance_Data - INFO - Cloud Score+ pixel threshold: 0.7
2026-11-10 15:58:44,209 - Reflectance_Data - INFO - Detailed statistics will not be computed
2026-11-10 15:58:44,210 - Reflectance_Data - INFO - Using Sentinel-2 Level 2A SR data.
2026-11-10 15:58:44,212 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-10 15:58:44,213 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)


Fetching Sentinel-2 collection...


2026-11-10 15:58:44,917 - final_Image - INFO - Creating median composite from 37 images
2026-11-10 15:58:44,918 - final_Image - INFO - Composite clipped to AOI
2026-11-10 15:58:46,263 - final_Image - INFO - Composite created from 2021-02-11 to 2021-12-18
2026-11-10 15:58:47,148 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-11-10 15:58:47,150 - Reflectance_Data - INFO - Date range: 2021-01-01 to 2021-12-31
2026-11-10 15:58:47,151 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2026-11-10 15:58:47,151 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-11-10 15:58:47,152 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-10 15:58:47,154 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)


Sentinel-2 Bands: ['BLUE', 'GREEN', 'RED', 'RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'NIR', 'RED_EDGE4', 'SWIR1', 'SWIR2']

Fetching Landsat 8 collection...


2026-11-10 15:58:48,513 - final_Image - INFO - Creating median composite from 3 images
2026-11-10 15:58:48,515 - final_Image - INFO - Composite clipped to AOI
2026-11-10 15:58:50,488 - final_Image - INFO - Composite created from 2021-05-09 to 2021-10-16


Landsat 8 Bands: ['AEROSOL', 'BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']


---
## 5. Individual Predictor Components
Standalone computation of each predictor component:
1. **Spectral Indices** (`SpectralCalculator`)
2. **Terrain Predictors** (`terrain_calculator`)
3. **Radar SAR Polarizations** (`sar.py`)

### 5.1 Spectral Indices Calculation
Computing standard indices alongside sensor-specific indices:
- `RENDVI` and `MCARI` computed on Sentinel-2
- `DBSI` and `MBI` computed on Landsat-8

In [5]:
# Sentinel-2 Indices (Testing Red-Edge index: RENDVI, MCARI)
s2_calc = SpectralCalculator()
s2_indices = s2_calc.calculate_indices(
    image=s2_composite,
    index_list=['NDVI', 'EVI', 'MNDWI', 'RENDVI', 'MCARI'],
    sensor='SENTINEL'
)
print("Sentinel-2 Computed Indices:", s2_indices.bandNames().getInfo())

# Landsat-8 Indices (Testing Soil and Water: DBSI, MBI, MNDWI)
l8_calc = SpectralCalculator()
l8_indices = l8_calc.calculate_indices(
    image=l8_composite,
    index_list=['NDVI', 'MNDWI', 'DBSI', 'MBI'],
    sensor='LANDSAT'
)
print("Landsat-8 Computed Indices:", l8_indices.bandNames().getInfo())

2026-11-10 15:58:55,426 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-11-10 15:58:56,111 - luma_ge.predictor - INFO - Retrieved default coefficients for EVI: {'g': 2.5, 'C1': 6.0, 'C2': 7.5, 'L': 1.0}
2026-11-10 15:58:58,592 - luma_ge.predictor - INFO - SpectralCalculator initialized


Sentinel-2 Computed Indices: ['NDVI', 'EVI', 'MNDWI', 'RENDVI', 'MCARI']
Landsat-8 Computed Indices: ['NDVI', 'MNDWI', 'DBSI', 'MBI']


### 5.2 Terrain Predictors (Topography & Geomorphology)
Using `terrain_calculator` to extract Elevation, Slope, Aspect, and morphological Landform classes.

In [6]:
tc = terrain_calculator()

# 1. Elevation (from NASADEM)
dem_elevation = tc.calculate_elevation(aoi)
print("Elevation Bands:", dem_elevation.bandNames().getInfo())

# 2. Morphological Landforms (Penumbral, Ridges, Valleys, Plains)
landform_image = tc.get_landform(aoi)
print("Landform Bands:", landform_image.bandNames().getInfo())

2026-11-10 15:59:01,707 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-11-10 15:59:01,710 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-10 15:59:01,711 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-10 15:59:02,133 - luma_ge.predictor - INFO - Retrieving landform layer from CSP/ERGo/1_0/Global/ALOS_landforms...
2026-11-10 15:59:02,134 - luma_ge.predictor - INFO - Successfully retrieved landform layer using ALOS source


Elevation Bands: ['elevation']
Landform Bands: ['landform']


### 5.3 Radar SAR Backscatter (Sentinel-1 & ALOS PALSAR)
Using `get_sar_data()` to extract C-band ($VV$, $VH$) and L-band ($HH$, $HV$).

In [7]:
sar_service = get_sar_data()

# Sentinel-1 C-Band
s1_sar = sar_service.get_sentinel1(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    orbit_pass='BOTH',
    reducer='median',
    include_ratio=True
)
print("Sentinel-1 SAR Bands:", s1_sar.bandNames().getInfo())

# ALOS PALSAR L-Band (with automatic gap fallback)
palsar_sar = sar_service.get_palsar(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    allow_nearest_year_fallback=True
)
print("ALOS PALSAR SAR Bands:", palsar_sar.bandNames().getInfo())

2026-11-10 15:59:05,341 - luma_ge.sar - INFO - get_sar_data initialized
2026-11-10 15:59:05,699 - luma_ge.sar - INFO - Sentinel-1: 51 scenes for 2021-01-01 to 2021-12-31 (BOTH)


Sentinel-1 SAR Bands: ['S1_VV', 'S1_VH', 'S1_VV_VH']


2026-11-10 15:59:06,641 - luma_ge.sar - INFO - PALSAR: 1 yearly mosaic scenes retrieved for years [2021]


ALOS PALSAR SAR Bands: ['PALSAR_HH', 'PALSAR_HV', 'PALSAR_HH_HV']


---
## 6. End-to-End Pipeline Orchestration: `compute_predictors()`
This is the primary method called by the backend service (`lulc_map.py` and `thematic.py`).

Notice that the configuration dynamically pulls **all registered spectral indices directly from `PredictorCatalog`**, so any new index added to `predictor.py` is automatically computed and tested here!

In [9]:
# Dynamically pull all registered spectral indices from the catalog
all_indices = [
    name for name, meta in predictor_catalog.get_all_catalog().items()
    if meta.get("type") == "spectral"
]

print(f"Testing with all {len(all_indices)} registered spectral indices:")
print(all_indices)

# Define full predictor configuration (matching backend payload schema)
predictor_config = {
    "individual_predictors": {
        "elevation": True,
        "slope": True,
        "aspect": True,
        "landform": True,
        "spectral_indices": all_indices,
        "sentinel1": True,
        "palsar": True
    }
}

orchestrator = PredictorCalculation()

# Execute on Sentinel-2
print("\n=== Running compute_predictors on Sentinel-2 ===")
s2_result = orchestrator.compute_predictors(
    composite=s2_composite,
    aoi=aoi,
    predictor_config=predictor_config,
    collection=s2_col
)

print(f"Status: Success={s2_result['success']}")
print(f"Detected Sensor: {s2_result['sensor']}")
print(f"Total Output Bands: {s2_result['total_bands']}")
print("Output Band Names:")
print(s2_result['stacked_predictors'].bandNames().getInfo())

if s2_result['warnings']:
    print("\nExpected Warnings (graceful skipping of Landsat-only indices):")
    for w in s2_result['warnings']:
        print(f"  - {w}")

2026-11-10 15:59:26,181 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-11-10 15:59:26,182 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-11-10 15:59:26,183 - luma_ge.sar - INFO - get_sar_data initialized
2026-11-10 15:59:26,183 - luma_ge.predictor - INFO - Predictor Calculation initialized with terrain, spectral and SAR calculators


Testing with all 19 registered spectral indices:
['NDVI', 'EVI', 'SAVI', 'MSAVI', 'GNDVI', 'MCARI', 'RENDVI', 'MCARI1', 'MNDWI', 'NDMI', 'AWEIsh', 'AWEInsh', 'NDTI', 'ANDWI', 'NDBI', 'DBSI', 'BLFEI', 'MBI', 'EBBI']

=== Running compute_predictors on Sentinel-2 ===


2026-11-10 15:59:27,125 - luma_ge.predictor - INFO - Prerequisite validation completed: valid=True
2026-11-10 15:59:27,917 - luma_ge.predictor - INFO - Predictor computation utilizing detected sensor: SENTINEL
2026-11-10 15:59:28,584 - luma_ge.predictor - INFO - Added 10 original multispectral bands
2026-11-10 15:59:28,586 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-10 15:59:28,588 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-10 15:59:28,588 - luma_ge.predictor - INFO - Successfully computed elevation
2026-11-10 15:59:28,589 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-11-10 15:59:28,589 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-10 15:59:28,590 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-10 15:59:28,591 - luma_ge.predictor - INFO - Successfully calculated slope layer u

Status: Success=True
Detected Sensor: SENTINEL
Total Output Bands: 38
Output Band Names:
['BLUE', 'GREEN', 'RED', 'RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'NIR', 'RED_EDGE4', 'SWIR1', 'SWIR2', 'elevation', 'slope', 'aspect', 'landform', 'NDVI', 'EVI', 'SAVI', 'MSAVI', 'GNDVI', 'MCARI', 'RENDVI', 'MCARI1', 'MNDWI', 'NDMI', 'AWEIsh', 'AWEInsh', 'NDTI', 'ANDWI', 'NDBI', 'DBSI', 'BLFEI', 'MBI', 'S1_VV', 'S1_VH', 'S1_VV_VH', 'PALSAR_HH', 'PALSAR_HV', 'PALSAR_HH_HV']


In [10]:
# Execute on Landsat 8
print("\n=== Running compute_predictors on Landsat-8 ===")
l8_result = orchestrator.compute_predictors(
    composite=l8_composite,
    aoi=aoi,
    predictor_config=predictor_config,
    collection=l8_col
)

print(f"Status: Success={l8_result['success']}")
print(f"Detected Sensor: {l8_result['sensor']}")
print(f"Total Output Bands: {l8_result['total_bands']}")
print("Output Band Names:")
print(l8_result['stacked_predictors'].bandNames().getInfo())

if l8_result['warnings']:
    print("\nExpected Warnings (graceful skipping of Sentinel-only indices):")
    for w in l8_result['warnings']:
        print(f"  - {w}")


=== Running compute_predictors on Landsat-8 ===


2026-11-10 15:59:41,108 - luma_ge.predictor - INFO - Prerequisite validation completed: valid=True
2026-11-10 15:59:41,421 - luma_ge.predictor - INFO - Predictor computation utilizing detected sensor: LANDSAT
2026-11-10 15:59:41,811 - luma_ge.predictor - INFO - Added 7 original multispectral bands
2026-11-10 15:59:41,813 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-10 15:59:41,813 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-10 15:59:41,814 - luma_ge.predictor - INFO - Successfully computed elevation
2026-11-10 15:59:41,815 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-11-10 15:59:41,816 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-10 15:59:41,817 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-10 15:59:41,818 - luma_ge.predictor - INFO - Successfully calculated slope layer usi

Status: Success=True
Detected Sensor: LANDSAT
Total Output Bands: 33
Output Band Names:
['AEROSOL', 'BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'elevation', 'slope', 'aspect', 'landform', 'NDVI', 'EVI', 'SAVI', 'MSAVI', 'GNDVI', 'MCARI1', 'MNDWI', 'NDMI', 'AWEIsh', 'AWEInsh', 'NDTI', 'ANDWI', 'NDBI', 'DBSI', 'BLFEI', 'MBI', 'S1_VV', 'S1_VH', 'S1_VV_VH', 'PALSAR_HH', 'PALSAR_HV', 'PALSAR_HH_HV']


---
## 7. Interactive Multi-Layer Visualization (`geemap`)
Visual inspection of predictors over the AOI to verify radiometric quality, spatial resolution alignment, and edge contrasts.

In [11]:
# Initialize Geemap canvas centered on AOI
Map = geemap.Map()
Map.centerObject(aoi, 11)

# 1. Base Optical Imagery (Natural Color & False Color NIR)
s2_stacked = s2_result['stacked_predictors']
Map.addLayer(
    s2_stacked,
    {'bands': ['RED', 'GREEN', 'BLUE'], 'min': 0, 'max': 3000},
    '1. S2 True Color (RGB)'
)
Map.addLayer(
    s2_stacked,
    {'bands': ['NIR', 'RED', 'GREEN'], 'min': 0, 'max': 4500},
    '2. S2 False Color Infrared (CIR)'
)

# 3. Spectral Indices
# NDVI (Vegetation Greenness)
ndvi_vis = {'bands': ['NDVI'], 'min': 0.0, 'max': 0.85, 'palette': ['blue', 'white', 'green']}
Map.addLayer(s2_stacked, ndvi_vis, '3. NDVI (Vegetation)')

# MNDWI (Water / Moisture)
mndwi_vis = {'bands': ['MNDWI'], 'min': -0.5, 'max': 0.5, 'palette': ['brown', 'yellow', 'cyan', 'blue']}
Map.addLayer(s2_stacked, mndwi_vis, '4. MNDWI (Water/Moisture)')

# RENDVI (Sentinel-2 Red Edge NDVI)
rendvi_vis = {'bands': ['RENDVI'], 'min': 0.0, 'max': 0.7, 'palette': ['red', 'yellow', 'darkgreen']}
Map.addLayer(s2_stacked, rendvi_vis, '5. RENDVI (Red Edge)')

# 4. Topography & Terrain
elev_vis = {'bands': ['elevation'], 'min': 0, 'max': 150, 'palette': ['darkblue', 'teal', 'lightgreen', 'brown', 'white']}
Map.addLayer(s2_stacked, elev_vis, '6. NASADEM Elevation')

landform_vis = {'bands': ['landform'], 'min': 11, 'max': 42, 'palette': ['141414', '383838', '808080', 'EBEBEB']}
Map.addLayer(s2_stacked, landform_vis, '7. Morphological Landforms')

# 5. Radar Backscatter (Sentinel-1 SAR)
s1_sar_vis = {'bands': ['S1_VV', 'S1_VH', 'S1_VV_VH'], 'min': [-18, -25, 0.2], 'max': [-4, -10, 0.9]}
Map.addLayer(s2_stacked, s1_sar_vis, '8. Sentinel-1 SAR (VV/VH/Ratio)')

# Add AOI boundary outline
aoi_outline = ee.Image().paint(ee.FeatureCollection([ee.Feature(aoi)]), color=1, width=2)
Map.addLayer(aoi_outline, {'palette': ['FF0000']}, 'AOI Boundary')

# Render the interactive map
Map

Map(center=[-2.9500033458894017, 104.7499999999997], controls=(WidgetControl(options=['position', 'transparent…

---
## 8. Technical Specification & Contract for Backend & Frontend

### 8.1 Backend Implementation Contract (`luma-be`)
When the frontend submits the predictor configuration, the payload structured below must be stored in `luma.predictor_config` and passed into `compute_predictors()`:

```json
{
  "individual_predictors": {
    "elevation": true,
    "slope": true,
    "aspect": true,
    "landform": true,
    "spectral_indices": ['NDVI', 'EVI', 'SAVI', 'MSAVI', 'GNDVI', 'MCARI', 'RENDVI', 'MCARI1', 'MNDWI', 'NDMI', 'AWEIsh', 'AWEInsh', 'NDTI', 'ANDWI', 'NDBI', 'DBSI', 'BLFEI', 'MBI', 'BI', 'EBBI'],
    "sentinel1": true,
    "palsar": false
  }
}
```

> **Important**: The backend whitelist in `application/logic/luma/predictor.py` (`set_predictor_config()`) should accept all predictors defined in `PredictorCatalog.get_all_catalog().keys()`.

### 8.2 Frontend Layer Styling Guide (`luma-fe`)
Recommended layer visualization palettes for frontend map rendering:

| Predictor Type | Band Name | Min | Max | Recommended Palette |
| :--- | :--- | :--- | :--- | :--- |
| **Vegetation** | `NDVI`, `EVI`, `GNDVI` | `0.0` | `0.85` | `['#0000FF', '#FFFFFF', '#00AA00']` |
| **Red Edge** | `RENDVI`, `MCARI` | `0.0` | `0.70` | `['#FF0000', '#FFFF00', '#006400']` |
| **Water** | `MNDWI`, `NDMI`, `AWEIsh` | `-0.5` | `0.50` | `['#8B4513', '#FFFFE0', '#00FFFF', '#0000FF']` |
| **Built-up / Soil** | `EBBI`, `DBSI`, `BLFEI`, `MBI`, `BI` | `-0.5` | `0.80` | `['#228B22', '#FFFF00', '#FF0000', '#800080']` |
| **Elevation** | `elevation` | `0` | `2000` | `['#00008B', '#008080', '#90EE90', '#8B4513', '#FFFFFF']` |
| **SAR Radar** | `S1_VV` / `S1_VH` | `-25 dB` | `0 dB` | Grayscale `['#000000', '#FFFFFF']` |